In [3]:
%load_ext autoreload
%autoreload 2
%env CUDA_VISIBLE_DEVICES=1

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
env: CUDA_VISIBLE_DEVICES=1


In [4]:
from lm_saes import SparseAutoEncoder, LowRankSparseAttention
from transformer_lens import HookedTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
from IPython.display import SVG, display

import torch
import chess

import json
from pathlib import Path
import os
import sys
from typing import Optional, Dict, Any, Tuple, List, Set

project_root = Path.cwd().parent.parent
sys.path.append(str(project_root))


from datasets import load_from_disk
import random


import numpy as np
from tqdm.auto import tqdm

from src.chess_utils import get_feature_vector, get_pos_from_square, get_piece_type_pos, feature_frequency_with_piece_type

In [5]:
model_name = 'lc0/BT4-1024x15x32h'
model = HookedTransformer.from_pretrained_no_processing(
    model_name,
    dtype=torch.float32,
).eval()

transcoders = {
    layer: SparseAutoEncoder.from_pretrained(
        f'/inspire/hdd/global_user/hezhengfu-240208120186/rlin_projects/rlin_projects/chess-SAEs-N/result_BT4/tc/k_30_e_16/L{layer}',
        dtype=torch.float32,
        device='cuda',
    )
    for layer in range(15)
}

lorsas = [
    LowRankSparseAttention.from_pretrained(
        f'/inspire/hdd/global_user/hezhengfu-240208120186/rlin_projects/rlin_projects/chess-SAEs-N/result_BT4/lorsa/k_30_e_16/L{layer}',
        dtype=torch.float32,
        device='cuda',
    )
    for layer in range(15)
]


state_dict keys: dict_keys(['embed.ma_gating_mul', 'embed.ma_gating_add', 'embed.ffn_alpha', 'embed.embedding_preprocess.weight', 'embed.embedding_preprocess.bias', 'embed.main_linear.weight', 'embed.main_linear.bias', 'embed.ffn_dense1.weight', 'embed.ffn_dense1.bias', 'embed.ffn_dense2.weight', 'embed.ffn_dense2.bias', 'embed.ln.weight', 'embed.ln.bias', 'embed.ln2.weight', 'embed.ln2.bias', 'blocks.0.alpha_input', 'blocks.0.alpha_out1', 'blocks.0.mha.qk_scale', 'blocks.0.mha.q_proj.weight', 'blocks.0.mha.q_proj.bias', 'blocks.0.mha.k_proj.weight', 'blocks.0.mha.k_proj.bias', 'blocks.0.mha.v_proj.weight', 'blocks.0.mha.v_proj.bias', 'blocks.0.mha.out_proj.weight', 'blocks.0.mha.out_proj.bias', 'blocks.0.mha.smolgen.compress.weight', 'blocks.0.mha.smolgen.dense1.weight', 'blocks.0.mha.smolgen.dense1.bias', 'blocks.0.mha.smolgen.ln1.weight', 'blocks.0.mha.smolgen.ln1.bias', 'blocks.0.mha.smolgen.dense2.weight', 'blocks.0.mha.smolgen.dense2.bias', 'blocks.0.mha.smolgen.ln2.weight', 'blo

[fill_missing_keys] missing_keys: {'policy_head.promotion_weight', 'pos_embed.W_pos'}
Loaded pretrained model lc0/BT4-1024x15x32h into HookedTransformer


/inspire/hdd/global_user/hezhengfu-240208120186/rlin_projects/rlin_projects/chess-SAEs-N/.venv/lib/python3.11/site-packages/torch/distributed/checkpoint/state_dict_loader.py:153: UserWarning: torch.distributed is disabled, unavailable or uninitialized, assuming the intent is to load in a single process.
  warnings.warn(


In [6]:

dataset_path = "/inspire/hdd/global_user/hezhengfu-240208120186/data/rlin_data/Chess/chess_master_data"

dataset = load_from_disk(dataset_path)

dataset_size = len(dataset)
print(f"length of dataset:{dataset_size}")

random_indices = random.sample(range(dataset_size), min(500, dataset_size))
random_data = [dataset[i]['fen'] for i in random_indices]

for i, item in enumerate(random_data[:5]):
    print(f"Sample {i+1}: {item}")
    
print(f'{len(random_data) = }')

length of dataset:51541318
Sample 1: 3r4/p3r1pk/1p3p1p/1b6/3B1PPP/2R1P1K1/P7/2R5 b - - 0 39
Sample 2: 6k1/4qpp1/pp5p/4R3/1PQ1P3/7P/5PP1/6K1 b - - 0 36
Sample 3: 2krr3/p4pp1/1pp1pn2/4b2p/P1PP2PP/2B2P2/1PK5/3R3R w - - 0 25
Sample 4: 2r1r1k1/pp2b1pp/1q2pn2/3pn1B1/8/2N3P1/PPQ1PPBP/R2R2K1 w - - 0 16
Sample 5: rnbqnrk1/pp2p2p/2p2ppb/2PpP3/3P1P2/1NN1B2P/PP1Q2P1/R3KB1R b KQ - 2 13
len(random_data) = 500


In [7]:
feature = (14, 1214, 'transcoder')

In [8]:
from src.chess_utils import get_move_from_policy_output_with_prob, get_start_end_pos_from_move_uci

layer, feature_id, feature_type = feature
print(f"验证feature: layer={layer}, feature_id={feature_id}, type={feature_type}")

activation_positions = []

for fen_idx, fen in enumerate(tqdm(random_data, desc="收集激活位置")):
    try:
        _, cache = model.run_with_cache(fen, prepend_bos=False)
        
        if feature_type == 'transcoder':
            hook_name = f'blocks.{layer}.resid_mid_after_ln'
            if hook_name not in cache:
                continue
            tc_input = cache[hook_name]
            activations = transcoders[layer].encode(tc_input)
            if activations.dim() == 3:
                acts_all = activations[0]
            else:
                acts_all = activations
        else:
            hook_name = f'blocks.{layer}.hook_attn_in'
            if hook_name not in cache:
                continue
            lorsa_input = cache[hook_name]
            activations = lorsas[layer].encode(lorsa_input)
            if activations.dim() == 3:
                acts_all = activations[0]
            else:
                acts_all = activations
        
        seq_len = acts_all.shape[0]
        feature_activations = acts_all[:, feature_id]
        
        for pos_idx in range(seq_len):
            act_value = feature_activations[pos_idx].item()
            if act_value > 0:
                activation_positions.append({
                    'fen_idx': fen_idx,
                    'fen': fen,
                    'pos_idx': pos_idx,
                    'activation_value': act_value,
                })
    except Exception as e:
        print(f"Error processing FEN {fen_idx}: {e}")
        continue

activation_positions.sort(key=lambda x: x['activation_value'], reverse=True)
top100 = activation_positions[:100]

print(f"\n收集到 {len(activation_positions)} 个激活位置，取top 100")
print(f"激活值范围: [{top100[-1]['activation_value']:.4f}, {top100[0]['activation_value']:.4f}]")

验证feature: layer=14, feature_id=1214, type=transcoder


收集激活位置:   0%|          | 0/500 [00:00<?, ?it/s]


收集到 570 个激活位置，取top 100
激活值范围: [1.2869, 2.4500]


In [9]:
matches = 0
total = 0

for item in tqdm(top100, desc="验证匹配"):
    try:
        fen = item['fen']
        pos_idx = item['pos_idx']
        
        output = model(fen, prepend_bos=False)
        logits = output[0]
        
        moves = get_move_from_policy_output_with_prob(logits, fen, return_list=True)
        if not moves:
            continue
        
        top_move_uci = moves[0][0]
        start_pos, end_pos = get_start_end_pos_from_move_uci(fen, top_move_uci)
        
        if end_pos == pos_idx:
            matches += 1
        
        total += 1
    except Exception as e:
        print(f"Error validating position: {e}")
        continue

match_ratio = matches / total if total > 0 else 0.0
print(f"\n验证结果:")
print(f"  总验证位置数: {total}")
print(f"  匹配终点位置数: {matches}")
print(f"  匹配率: {match_ratio:.2%}")
print(f"\n前10个激活位置的验证:")
for i, item in enumerate(top100[:10]):
    try:
        fen = item['fen']
        pos_idx = item['pos_idx']
        
        output = model(fen, prepend_bos=False)
        logits = output[0]
        
        moves = get_move_from_policy_output_with_prob(logits, fen, return_list=True)
        if moves:
            top_move_uci = moves[0][0]
            start_pos, end_pos = get_start_end_pos_from_move_uci(fen, top_move_uci)
            match = "✓" if end_pos == pos_idx else "✗"
            print(f"  {i+1}. pos={pos_idx}, move={top_move_uci}, end_pos={end_pos}, 激活值={item['activation_value']:.4f} {match}")
    except Exception as e:
        print(f"  {i+1}. Error: {e}")

验证匹配:   0%|          | 0/100 [00:00<?, ?it/s]


验证结果:
  总验证位置数: 100
  匹配终点位置数: 96
  匹配率: 96.00%

前10个激活位置的验证:
  1. pos=44, move=c4e6, end_pos=44, 激活值=2.4500 ✓
  2. pos=35, move=b3d5, end_pos=35, 激活值=2.3324 ✓
  3. pos=26, move=d6c5, end_pos=26, 激活值=2.2276 ✓
  4. pos=29, move=e5f4, end_pos=29, 激活值=2.2176 ✓
  5. pos=12, move=f8e7, end_pos=12, 激活值=2.2142 ✓
  6. pos=37, move=f5f4, end_pos=37, 激活值=2.1184 ✓
  7. pos=10, move=b2c2, end_pos=10, 激活值=2.0969 ✓
  8. pos=23, move=g2h3, end_pos=23, 激活值=2.0775 ✓
  9. pos=5, move=e8f8, end_pos=5, 激活值=2.0616 ✓
  10. pos=21, move=d1f3, end_pos=21, 激活值=2.0376 ✓
